In [ ]:
# Raymond needs these
!pip install textblob
!pip install textstat
!pip install pandas

# I think Raymond doesn't need these
#!pip install spacy
# in a terminal execute:
# python -m spacy download en_core_web_sm
# pip install -U datasets pandas

In [ ]:
import pandas as pd
from datasets import load_dataset
import textstat
from textblob import TextBlob
import spacy

## 0. First time loading the dataset from ShenLab

In [ ]:
# Load real dataset
real_ds = load_dataset(
    "ShenLab/MentalChat16K",
    data_files="Interview_Data_6K.csv",
    split="train"
)


In [ ]:
# Load synthetic datasets
synthetic_ds = load_dataset(
    "ShenLab/MentalChat16K",
    data_files="Synthetic_Data_10K.csv",
    split="train"
)



In [ ]:
# Convert both to pandas and add labels.
df_real = real_ds.to_pandas()
df_synth = synthetic_ds.to_pandas()

# Add labels
df_real["source"] = "real"
df_synth["source"] = "synthetic"


In [ ]:
 # Combine
df = pd.concat([df_real, df_synth], ignore_index=True)

df.head()

In [ ]:
df.tail()

#### Saving a copy of the dataset

In [ ]:
# keep the whole dataset in a file
df.to_csv("mentalchat16k.csv", index=False, encoding="utf-8")

## 1. Reading saved data from local CSV

Only need to start running code from here. 

In [ ]:
# load the local dataset as dataframe
df = pd.read_csv("mentalchat16k.csv", encoding="utf-8")
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

# 2. Computing linguistic features
## Readibility
- Flesch Reading Ease
- Gunning Fog Index
- SMOG Index
- Dale-Chall Index
## NLP Features
- Subjectivity
- Polarity (sentiment)

In [ ]:
#nlp = spacy.load("en_core_web_sm")

NEGATION_WORDS = {"not", "no", "never", "none", "n't", "cannot"}

In [ ]:
NEGATION_WORDS = {"not", "no", "never", "none", "n't", "cannot"}
HEDGING_WORDS = {"maybe", "perhaps", "might", "could", "possibly"}

def compute_features(text):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return {}

    # --- Readability ---
    features = {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "gunning_fog": textstat.gunning_fog(text),
        "smog": textstat.smog_index(text),
        "dale_chall": textstat.dale_chall_readability_score(text),
    }

    # --- Subjectivity and Polarity ---
    blob = TextBlob(text)
    features["subjectivity"] = blob.sentiment.subjectivity
    features["polarity"] = blob.sentiment.polarity  # ← no trailing comma

    # --- Negation ---
    tokens = text.lower().split()
    features["negation_count"] = sum(t in NEGATION_WORDS for t in tokens)
    features["has_negation"] = int(features["negation_count"] > 0)

    # --- Sentence Length ---
    sentences = text.split(".")
    words = text.split()
    features["avg_sentence_length"] = len(words) / max(len(sentences), 1)

    # --- Vocabulary Richness ---
    features["lexical_diversity"] = len(set(words)) / len(words) if len(words) > 0 else 0

    # --- Hedging ---
    features["hedging_count"] = sum(word in HEDGING_WORDS for word in words)

    return features

In [ ]:
# Testing code (using only 10 data points)

df_temp = df.head(10).copy()
features_df = df_temp["output"].apply(lambda x: compute_features(x)).apply(pd.Series)
df_temp = pd.concat([df_temp, features_df], axis=1)
df_temp

## 3. Applying language features to each row

In [ ]:
features_df = df["output"].apply(lambda x: compute_features(x)).apply(pd.Series)

df = pd.concat([df, features_df], axis=1)

In [ ]:
df.head(15)

## 4. Saving the output to a CSV file

In [ ]:
df.to_csv("mentalchat16k_metrics.csv", index=False, encoding="utf-8")

# Possible Research Questions for Considerations

## Research Questions (Choose 1 or more)

**RQ1.** Are there statistically significant differences between synthetic and interview-based responses across linguistic features (e.g., readability, lexical familiarity, subjectivity, and negation)?

**RQ2.** Which linguistic features most strongly distinguish synthetic responses from interview-based responses?

**RQ3.** Do synthetic responses systematically differ in ways that may affect downstream evaluation, such as readability, subjectivity, and structural complexity?

**RQ4.** To what extent can synthetic data serve as a proxy for interview-based data in linguistic analysis?


**Raymond, please add your own.**

## Other Linguistic Features to Consider (on your own exploration)

Length and structure features
- word_count 
- sentence_count
- char_count
- average words per sentence
- average chars per word

Lexical diversity features
- unique_word_count 

Confidence features
- hedge word count e.g., perhaps, maybe, could, somewhat, probably, likely, and many more
- pronoun feature or point of view type e.g., first person, second person, third person language
- question mark count
- ellipsis count
- sentiment score
  

## 5. Loading file with some initial metrics

In [ ]:
# load the local dataset as dataframe
df2 = pd.read_csv("mentalchat16k_metrics.csv", encoding="utf-8")
df2.head()

In [ ]:
df2.describe()

In [ ]:
df2.groupby('source')['flesch_reading_ease'].mean()

In [ ]:
# Removed the 3 outliers with negative flesch_reading ease

df2[df2['flesch_reading_ease'] >= 0].groupby('source')['flesch_reading_ease'].mean()

In [ ]:
df2.hist(figsize=(8,8), bins = 15)